# 🚀 Building Your First AI Agent with LangGraph - Step-by-Step Tutorial

## 👋 Welcome, Developer!

This tutorial walks you through creating a **travel planning AI agent** using **LangGraph** + **LangChain** + an **Azure AI Foundry (Azure OpenAI)** model. You'll learn how to assemble tools, prompts, state, and graph execution flow.

**What You'll Learn:**
- ✅ LangGraph vs traditional agent architecture
- ✅ How to define and register tools (abilities)
- ✅ How message state flows through a graph
- ✅ How to build a ReAct-style agent with `create_react_agent`
- ✅ How to integrate an Azure-hosted chat model
- ✅ How to run single and multi-turn interactions

**Prerequisites:**
- Basic Python knowledge
- Azure OpenAI / AI Foundry access (deployment like `gpt-4o-mini`)
- 15–20 minutes

Let's build your first LangGraph agent! 🌍

## 📚 Step 1: Understand LangGraph Architecture

LangGraph lets you define agent execution as a **graph of nodes** (LLM calls, tool use, routing conditions) operating over a shared **state object** (e.g. `messages`).

```text
┌─────────────────────────────────────────────────────────┐
│                    LANGGRAPH AGENT FLOW                  │
│  User Message → (Prompt + LLM Node) → Tool Decision → Tool Call → Response Merge │
│                               ↖ reasoning loop ↗         │
└─────────────────────────────────────────────────────────┘
                    State = { messages: [...] }
```

**Key Concepts:**
1. **State**: A Python dict / TypedDict that accumulates conversation messages.
2. **Nodes**: Functions or model calls that transform or extend the state.
3. **Tools**: Declared capabilities the LLM can invoke (wrapped as `Tool`).
4. **Graph Execution**: LangGraph coordinates loops (ReAct reasoning) & tool calls.
5. **Prompt + Messages**: Combined context passed to model each iteration.

**Flow (ReAct style):**
1. User asks for a plan.
2. Model considers available tools.
3. Decides to call tool → receives result.
4. Produces final itinerary message.
5. State updated with all intermediate steps.

We'll use a helper: `create_react_agent` to build this quickly.

## 📦 Step 2: Install Required Packages (Pinned Versions)

We use versions pinned in `requirements.txt` to ensure reproducibility:
- `langgraph==1.0.2`
- `langchain==1.0.3`
- `langchain-openai==1.0.2`
- `python-dotenv==1.0.1` (env management)
- `azure-identity==1.19.0` (optional advanced auth)

(If already installed, pip will skip reinstallation.)

In [ ]:
# Pinned installs based on repository requirements
! pip install langgraph==1.0.2 langchain==1.0.3 langchain-openai==1.0.2 python-dotenv==1.0.1 azure-identity==1.19.0 -q
print('✅ Dependencies installed (or already present).')

## 🔧 Step 3: Imports & Environment Setup

**Imports Overview:**
- `AzureChatOpenAI`: Azure-bound chat model client (OpenAI-compatible)
- `Tool`: Wrapper for callable functions the LLM can decide to use
- `HumanMessage`, `AIMessage`: Structured message objects
- `create_react_agent`: Helper that builds a reasoning loop + tool use agent
- `dotenv`: Load env vars from `.env` (endpoint, key, deployment)

Ensure your `.env` contains at minimum:
```env
AZURE_AI_FOUNDRY_ENDPOINT=https://your-resource.openai.azure.com
AZURE_AI_FOUNDRY_API_KEY=your_azure_openai_key
AZURE_AI_FOUNDRY_MODEL_ID=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-07-01-preview
```
(Adjust deployment name/version as needed.)

In [1]:
import os
from random import randint
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import create_react_agent

load_dotenv()
print('✅ Imports loaded & environment variables read.')

✅ Imports loaded & environment variables read.


## 🛠️ Step 4: Define a Custom Tool

Tools are structured capabilities the model can selectively invoke during reasoning.

**Design Principles:**
1. Descriptive `name` (used in model tool selection)
2. Clear docstring with purpose & parameters
3. Deterministic output (avoid unexpected side effects)
4. Return simple, serializable types (strings, numbers, JSON)

We'll create `get_random_destination` that returns one of ten curated travel spots.

In [2]:
def get_random_destination(*args, **kwargs) -> str:
    """Return a random travel destination from a curated list.
    
    This tool helps the agent pick a location when the user hasn't specified one.
    No inputs required.
    Returns: str - destination name like 'Tokyo, Japan'.
    """
    destinations = [
        'Barcelona, Spain', 'Paris, France', 'Berlin, Germany', 'Tokyo, Japan',
        'Sydney, Australia', 'New York, USA', 'Cairo, Egypt', 'Cape Town, South Africa',
        'Rio de Janeiro, Brazil', 'Bali, Indonesia'
    ]
    return destinations[randint(0, len(destinations) - 1)]

destination_tool = Tool(
    name='get_random_destination',
    func=get_random_destination,
    description='Returns the name of a random global travel destination.'
)

print('🎲 Sample destination:', get_random_destination())
print('✅ Tool registered.')

🎲 Sample destination: Tokyo, Japan
✅ Tool registered.


## 🤖 Step 5: Configure Azure Chat Model

We instantiate `AzureChatOpenAI` with endpoint, key, deployment, version.

**Important Variables:**
- `AZURE_AI_FOUNDRY_ENDPOINT` (resource or project endpoint)
- `AZURE_AI_FOUNDRY_API_KEY` (Azure OpenAI key)
- `AZURE_AI_FOUNDRY_MODEL_ID` (deployment name, e.g. `gpt-4o-mini`)
- `AZURE_OPENAI_API_VERSION` (matches REST version)

We'll also run a minimal test invocation to confirm connectivity before building the agent.

In [3]:
azure_endpoint = os.getenv('AZURE_AI_FOUNDRY_ENDPOINT')
azure_api_key = os.getenv('AZURE_AI_FOUNDRY_API_KEY')
azure_deployment = os.getenv('AZURE_AI_FOUNDRY_MODEL_ID') or os.getenv('AZURE_AI_FOUNDRY_MODEL')
api_version = os.getenv('AZURE_OPENAI_API_VERSION', '2024-07-01-preview')

print('🔍 Endpoint:', azure_endpoint)
print('🔍 Deployment:', azure_deployment)
print('🔍 API Version:', api_version)

if not all([azure_endpoint, azure_api_key, azure_deployment]):
    raise ValueError('Missing one or more required environment variables for Azure OpenAI.')

llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    azure_deployment=azure_deployment,
    api_version=api_version,
    temperature=0.7,
)

# Quick connectivity test
try:
    test_resp = llm.invoke([HumanMessage(content='Return just the word TEST')])
    print('✅ Model test OK ->', test_resp.content)
except Exception as e:
    print('❌ Model test failed:', e)
    raise

🔍 Endpoint: https://ibecfoundry.openai.azure.com/
🔍 Deployment: gpt-4o
🔍 API Version: 2024-02-01
✅ Model test OK -> TEST
✅ Model test OK -> TEST


## 📝 Step 6: Define System Instructions & Prompt

We craft a **system message** that establishes role, tasks, output format guidelines. LangGraph's `create_react_agent` accepts a prompt template including a `MessagesPlaceholder` for dynamic conversation state.

**Prompt Components:**
- System: Role + behavior + required steps
- MessagesPlaceholder: Injects ongoing conversation (state)

We'll require the agent to:
1. Use the destination tool if user didn't specify a location.
2. Produce structured itinerary sections.

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_INSTRUCTIONS = (
    'You are a helpful travel planning AI. When asked to plan a day trip, '
    '(1) pick a random destination using the get_random_destination tool if none provided, then '
    '(2) produce a structured itinerary with morning, afternoon, evening, local cuisine, and tips.'
)

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_INSTRUCTIONS),
    MessagesPlaceholder(variable_name='messages'),
])
print('✅ Prompt template constructed.')

✅ Prompt template constructed.


## 🎯 Step 7: Build the LangGraph ReAct Agent

`create_react_agent(model, tools, prompt)` wires together:
- LLM reasoning loop
- Tool invocation decisions
- Message state accumulation

Under the hood it builds a graph with conditional edges and iterates until the model produces a final answer (no further tool calls needed).

In [5]:
agent = create_react_agent(
    model=llm,
    tools=[destination_tool],
    prompt=prompt
)
print('✅ ReAct agent created. Tools available:', [t.name for t in [destination_tool]])

✅ ReAct agent created. Tools available: ['get_random_destination']


C:\Users\ivbeljan\AppData\Local\Temp\ipykernel_30008\310353585.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 💬 Step 8: Invoke the Agent (First Turn)

We pass an initial state: `{ 'messages': [HumanMessage(...)] }`. LangGraph will:
1. Combine system + user + prior messages.
2. Let model reason → decide on tool use.
3. Call tool if needed, append results.
4. Produce final itinerary.

We'll capture messages and extract the last `AIMessage` content.

In [7]:
user_request = 'Plan me a day trip'
result_state = agent.invoke({'messages': [HumanMessage(content=user_request)]})
messages = result_state.get('messages', [])
# Find last AI message
final_ai = next((m for m in reversed(messages) if isinstance(m, AIMessage)), None)

print('👤 User:', user_request)
print('🤖 Agent Response:')
print(final_ai.content if final_ai else 'No AI response found.')

👤 User: Plan me a day trip
🤖 Agent Response:
Here’s a day trip itinerary for Barcelona, Spain:

---

### Morning: Explore the Gothic Quarter
- Start your day in the historic **Barri Gòtic (Gothic Quarter)**, a maze of narrow medieval streets filled with charming shops and cafes.
- Visit the **Barcelona Cathedral**, a stunning Gothic structure.
- Stroll down **La Rambla**, the famous pedestrian street buzzing with activity.
- Stop by the **Mercat de Sant Josep de la Boqueria**, a vibrant food market, and grab a fresh juice or a snack.

---

### Afternoon: Gaudí's Masterpieces
- Head to **Sagrada Família**, Antoni Gaudí’s iconic basilica. Be sure to book tickets in advance to skip the lines.
- Afterward, take a short metro ride to **Park Güell**, another Gaudí creation. Enjoy the colorful mosaic art and sweeping views of the city.
- Have lunch at a nearby restaurant. Try local dishes like **paella** or **escalivada** (grilled vegetables).

---

### Evening: Sunset at Montjuïc
- Make your

## 📖 Step 9: Inspect Raw Message Trace

Inspecting the full `messages` list helps debugging tool decisions. Typically you'll see entries like:
- HumanMessage (user request)
- AIMessage (thought / tool call request)
- Tool invocation representation (inside intermediate AI messages)
- Final AIMessage (answer)

We'll print message role + truncated content.

In [8]:
for i, m in enumerate(messages):
    preview = (m.content[:120] + '...') if hasattr(m, 'content') and len(getattr(m, 'content', '')) > 120 else getattr(m, 'content', '')
    print(f'Message {i+1}: {m.__class__.__name__} -> {preview}')

Message 1: HumanMessage -> Plan me a day trip
Message 2: AIMessage -> 
Message 3: ToolMessage -> Barcelona, Spain
Message 4: AIMessage -> Here’s a day trip itinerary for Barcelona, Spain:

---

### Morning: Explore the Gothic Quarter
- Start your day in the ...


## 🔄 Step 10: Continue the Conversation (Second Turn)

LangGraph state can be reused by feeding prior messages back in. We'll ask for a different destination to test tool reuse.

Strategy: Start with previous `messages` + new HumanMessage appended.

In [9]:
follow_up = 'I prefer somewhere with beaches. Suggest another and plan a day.'
extended_state = {'messages': messages + [HumanMessage(content=follow_up)]}
result_state2 = agent.invoke(extended_state)
messages2 = result_state2.get('messages', [])
final_ai2 = next((m for m in reversed(messages2) if isinstance(m, AIMessage)), None)

print('👤 Follow-up:', follow_up)
print('🤖 Agent Response (Turn 2):')
print(final_ai2.content if final_ai2 else 'No AI response found.')
print(f'💬 Total messages after two turns: {len(messages2)}')

👤 Follow-up: I prefer somewhere with beaches. Suggest another and plan a day.
🤖 Agent Response (Turn 2):
Tokyo, Japan, isn't a traditional beach destination, but it is close to some wonderful coastal areas. For a beach-focused day trip, I suggest heading to **Enoshima**, a small island near Tokyo known for its beaches and scenic beauty. Here's your itinerary:

---

### Morning: Head to Enoshima Island
- Take the train from Tokyo to **Enoshima** (approximately 1 hour via the Odakyu Line or JR line).
- Start your visit at the **Katase Higashihama Beach**, a popular spot for swimming and relaxing on the sandy shore.
- Grab a coffee or light breakfast at one of the beachside cafes.

---

### Afternoon: Explore Enoshima Island
- Walk across the **Enoshima Benten Bridge** to the island.
- Visit the **Enoshima Shrine**, dedicated to the goddess Benzaiten.
- Explore the **Enoshima Iwaya Caves**, natural caves formed by coastal erosion.
- Enjoy lunch at a seafood restaurant on the island. Try *

## 🧠 Step 11: Execution Breakdown

### Turn 1 ("Plan me a day trip")
1. Model received system + user message.
2. Determined destination missing → tool call.
3. Tool returned random location.
4. Model composed structured itinerary.

### Turn 2 (Preference added)
1. Prior messages re-provided → model had context.
2. Considered beach preference (semantic hint).
3. Re-used destination tool to fetch another location.
4. Generated beach-oriented itinerary.

### Why LangGraph?
- Explicit state makes multi-step flows inspectable.
- Graph extensibility: add memory nodes, validation, routing easily.
- Built-in support for streaming and checkpointing (advanced).

### Tool Usage Insights
- Model chooses tool based on description relevance.
- Clear naming + description improves accuracy.

### Next Extensibility Ideas
- Add sentiment node before choosing activities.
- Insert retrieval node for local tips (RAG).
- Add guardrail node (budget or safety constraints).

## 🚀 Step 12: Optional Streaming (Debugging)

Streaming shows intermediate reasoning/tool usage as updates. Helpful for transparency & UI integration.
We'll request a food-focused plan and stream progress.

In [13]:
# Display the graph structure and execution flow
print('🔍 Graph Structure Visualization:')
print('=' * 50)

# Display the graph as ASCII representation
try:
    graph_ascii = agent.get_graph().draw_ascii()
    print(graph_ascii)
except Exception as e:
    print(f'ASCII visualization not available: {e}')
    


🔍 Graph Structure Visualization:
ASCII visualization not available: Install grandalf to draw graphs: `pip install grandalf`.
